# ReasonIF base vs DPO-only vs SFT-to-DPO viewer (8192 tokens)

Browse the base, DPO-only, and SFT-to-DPO responses one at a time. The viewer preserves the same question position when switching models.

In [ ]:
from pathlib import Path
import json

RUN_ROOT = (
    Path.home()
    / "Downloads/CoT_Controllability_outputs/reasonif_three_model_8192_comparison"
    / "n300_seed42_new8192_temp0.1"
)
RAW_PATHS = {
    "Base Qwen3-0.6B": RUN_ROOT / "base_qwen3_0_6b_raw.jsonl",
    "DPO only (2 epochs)": RUN_ROOT / "dpo_only_2_epochs_raw.jsonl",
    "Final SFT -> DPO": RUN_ROOT / "final_sft_dpo_raw.jsonl",
}
SCORED_PATH = RUN_ROOT / "scored_responses.jsonl"

def load_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    rows = []
    # Split JSONL only on literal LF; U+2028 may legally occur in JSON text.
    for line_number, line in enumerate(path.read_text(encoding="utf-8").split("\n"), 1):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError as error:
            raise ValueError(f"Invalid JSONL at {path}:{line_number}: {error}") from error
    return rows

score_lookup = {}
if SCORED_PATH.is_file():
    for row in load_jsonl(SCORED_PATH):
        label = row.get("model", row.get("model_label"))
        score_lookup[(label, row["dataset_index"])] = row

response_sets = {}
for label, path in RAW_PATHS.items():
    rows = load_jsonl(path)
    merged = []
    for row in rows:
        scored = score_lookup.get((label, row["dataset_index"]), {})
        merged.append({**row, **scored})
    response_sets[label] = sorted(merged, key=lambda row: row["dataset_index"])
    print(f"{label}: {len(merged)} responses")
    print("  Raw:", path)
print("Scores:", SCORED_PATH if SCORED_PATH.is_file() else "not available")

In [ ]:
import html
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

model_selector = widgets.Dropdown(
    options=list(response_sets),
    value="Base Qwen3-0.6B",
    description="Model",
    layout=widgets.Layout(width="290px"),
)
responses = response_sets[model_selector.value]
previous_button = widgets.Button(description="Previous")
next_button = widgets.Button(description="Next")
position = widgets.BoundedIntText(value=1, min=1, max=len(responses), description="Response")
jump_button = widgets.Button(description="Go")
counter = widgets.HTML()
details = widgets.Output()

def shown(value):
    value = "" if value is None else str(value)
    return value if value.strip() else "*(empty)*"

def render(index):
    index = max(0, min(index, len(responses) - 1))
    position.value = index + 1
    row = responses[index]
    previous_button.disabled = index == 0
    next_button.disabled = index == len(responses) - 1
    follows = row.get("instruction_following")
    correct = row.get("answer_correct")
    counter.value = (
        f"<b>{index + 1} / {len(responses)}</b> &nbsp; "
        f"dataset_index={row.get('dataset_index')} &nbsp; "
        f"source={html.escape(str(row.get('source', '')))} &nbsp; "
        f"truncated={bool(row.get('truncated'))}"
    )
    score_lines = [
        f"- **Reference answer:** `{row.get('answer', '')}`",
        f"- **Predicted answer:** `{row.get('predicted_answer', '')}`",
        f"- **Instruction following:** `{follows if follows is not None else 'not scored'}`",
        f"- **Answer correct:** `{correct if correct is not None else 'not scored'}`",
        f"- **Constraint:** `{row.get('constraint_name', row.get('constraint', ''))}`",
        f"- **Output tokens:** `{row.get('output_tokens', '')}`",
        f"- **Truncated:** `{row.get('truncated', '')}`",
        f"- **Batch seconds:** `{row.get('batch_seconds', '')}`",
    ]
    with details:
        clear_output(wait=True)
        display(Markdown(
            "### Prompt\n\n" + shown(row.get("prompt")) +
            "\n\n---\n\n### Reasoning trace\n\n" + shown(row.get("reasoning_content")) +
            "\n\n---\n\n### Final response\n\n" + shown(row.get("content")) +
            "\n\n---\n\n### Scoring\n\n" + "\n".join(score_lines) +
            "\n\n---\n\n<details><summary>Raw model output</summary>\n\n" +
            shown(row.get("raw_output")) + "\n\n</details>"
        ))

def change_model(change):
    global responses
    if change.get("name") != "value":
        return
    current_position = position.value
    responses = response_sets[change["new"]]
    position.max = len(responses)
    render(min(current_position, len(responses)) - 1)

model_selector.observe(change_model, names="value")
previous_button.on_click(lambda _: render(position.value - 2))
next_button.on_click(lambda _: render(position.value))
jump_button.on_click(lambda _: render(position.value - 1))
controls = widgets.HBox([model_selector, previous_button, next_button, position, jump_button, counter])
display(widgets.VBox([controls, details]))
render(0)